In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import types

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/08 11:22:19 WARN Utils: Your hostname, boyka.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.3 instead (on interface en0)
26/03/08 11:22:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/08 11:22:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Question 1: Install Spark and PySpark

In [3]:
spark.version

'4.1.1'

## Question 2: Yellow November 2025

In [4]:
!ls 

Homework.ipynb data           homework.md


In [5]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-08 11:22:29--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 13.227.235.199, 13.227.235.181, 13.227.235.205, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|13.227.235.199|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet’

yellow_tripdata_202 100%[===================>]  67.84M  63.1MB/s    in 1.1s    

2026-03-08 11:22:30 (63.1 MB/s) - ‘yellow_tripdata_2025-11.parquet’ saved [71134255/71134255]



In [6]:
!ls -lh yellow_tripdata_2025-11.parquet

-rw-r--r--@ 1 boyka  staff    68M Dec 19 22:51 yellow_tripdata_2025-11.parquet


In [7]:
df = spark.read.parquet('yellow_tripdata_2025-11.parquet')

In [8]:
type(df)

pyspark.sql.classic.dataframe.DataFrame

In [9]:
df.show(1)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [10]:
df = df.repartition(4)

In [11]:
!ls

Homework.ipynb                  homework.md
data                            yellow_tripdata_2025-11.parquet


In [12]:
df.write.parquet('data/pq/hw/2025/11/', compression="none", mode="overwrite")

In [13]:
!ls

Homework.ipynb                  homework.md
data                            yellow_tripdata_2025-11.parquet


In [14]:
!ls -lh data/pq/hw/2025/11

total 229888
-rw-r--r--@ 1 boyka  staff     0B Mar  8 11:23 _SUCCESS
-rw-r--r--@ 1 boyka  staff    27M Mar  8 11:23 part-00000-0888ded3-3775-4a29-ae90-9c5622072d90-c000.parquet
-rw-r--r--@ 1 boyka  staff    27M Mar  8 11:23 part-00001-0888ded3-3775-4a29-ae90-9c5622072d90-c000.parquet
-rw-r--r--@ 1 boyka  staff    27M Mar  8 11:23 part-00002-0888ded3-3775-4a29-ae90-9c5622072d90-c000.parquet
-rw-r--r--@ 1 boyka  staff    27M Mar  8 11:23 part-00003-0888ded3-3775-4a29-ae90-9c5622072d90-c000.parquet


In [15]:
from pyspark.sql import functions as F

## Question 3: Count records


In [16]:
df \
    .withColumn('pickup_datetime', F.to_date(df.tpep_pickup_datetime)) \
    .filter("pickup_datetime = '2025-11-15'") \
    .count()

162604

In [17]:
df.createOrReplaceTempView('yellow_tripdata_2025_11')

In [18]:
spark.sql("""
SELECT
    COUNT(1)
FROM 
    yellow_tripdata_2025_11
WHERE
    to_date(tpep_pickup_datetime) = '2025-11-15';
""").show()

+--------+
|count(1)|
+--------+
|  162604|
+--------+



In [19]:
df.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee',
 'cbd_congestion_fee']

## Question 4: Longest trip


In [20]:
df \
    .withColumn('duration', (F.unix_timestamp(df.tpep_dropoff_datetime) - F.unix_timestamp(df.tpep_pickup_datetime)) / 3600) \
    .select("duration") \
    .orderBy('duration', ascending=False) \
    .limit(5) \
    .show()

[Stage 17:=============================>                            (4 + 4) / 8]

+-----------------+
|         duration|
+-----------------+
|90.64666666666666|
|76.94833333333334|
|76.21388888888889|
|69.28861111111111|
|67.08055555555555|
+-----------------+



In [21]:
spark.sql("""
SELECT
    MAX((unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 3600) AS duration
FROM 
    yellow_tripdata_2025_11
ORDER BY
    1 DESC
LIMIT 10;
""").show()

+-----------------+
|         duration|
+-----------------+
|90.64666666666666|
+-----------------+



## Question 6: Least frequent pickup location zone


In [22]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-08 11:23:49--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 13.227.235.160, 13.227.235.205, 13.227.235.199, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|13.227.235.160|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0.001s  

2026-03-08 11:23:49 (17.7 MB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [23]:
!ls

Homework.ipynb                  taxi_zone_lookup.csv
data                            yellow_tripdata_2025-11.parquet
homework.md


In [24]:
df.schema

StructType([StructField('VendorID', IntegerType(), True), StructField('tpep_pickup_datetime', TimestampNTZType(), True), StructField('tpep_dropoff_datetime', TimestampNTZType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('RatecodeID', LongType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('payment_type', LongType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('Airport_fee', DoubleType(), True), StructField('cbd_congestio

In [25]:
zones = spark.read.csv('taxi_zone_lookup.csv')

In [26]:
zones.show(1)

+----------+-------+----+------------+
|       _c0|    _c1| _c2|         _c3|
+----------+-------+----+------------+
|LocationID|Borough|Zone|service_zone|
+----------+-------+----+------------+
only showing top 1 row


In [27]:
schema = types.StructType([
    types.StructField('LocationID', types.IntegerType(), True),
    types.StructField('Borough', types.StringType(), True),
    types.StructField('Zone', types.StringType(), True),
    types.StructField('service_zone', types.StringType(), True),
])

In [28]:
df_zones = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv('taxi_zone_lookup.csv')

In [29]:
df_zones.schema

StructType([StructField('LocationID', IntegerType(), True), StructField('Borough', StringType(), True), StructField('Zone', StringType(), True), StructField('service_zone', StringType(), True)])

In [30]:
df_zones.show(1)

+----------+-------+--------------+------------+
|LocationID|Borough|          Zone|service_zone|
+----------+-------+--------------+------------+
|         1|    EWR|Newark Airport|         EWR|
+----------+-------+--------------+------------+
only showing top 1 row


In [31]:
!ls

Homework.ipynb                  taxi_zone_lookup.csv
data                            yellow_tripdata_2025-11.parquet
homework.md


In [32]:
df_zones.columns

['LocationID', 'Borough', 'Zone', 'service_zone']

In [33]:
df_zones.createOrReplaceTempView('zones')

In [34]:
spark.sql("""
SELECT
    pul.Zone,
    COUNT(1)
FROM 
    yellow_tripdata_2025_11 tripdata LEFT JOIN zones pul ON tripdata.PULocationID = pul.LocationID
GROUP BY 
    1
ORDER BY
    2 ASC
LIMIT 5;
""").show(truncate=False)

+---------------------------------------------+--------+
|Zone                                         |count(1)|
+---------------------------------------------+--------+
|Eltingville/Annadale/Prince's Bay            |1       |
|Governor's Island/Ellis Island/Liberty Island|1       |
|Arden Heights                                |1       |
|Port Richmond                                |3       |
|Green-Wood Cemetery                          |4       |
+---------------------------------------------+--------+

